In [99]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "sanchez2017chimpanzees")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Sanchez_2017_Dataset_Study1a.csv")
complete_path_2 = os.path.join(original_data_pathway, "Sanchez_2017_Dataset_Study 2.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [100]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)

df1['experiment_name']="1a"
df2['experiment_name']="2"

df1 = df1.rename(columns={"Subject Left Pull": "subject_left_pull_latency", 
    "Subject Right Pull": "subject_right_pull_latency"})




In [101]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
data_frames=[ df1, df2]

for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s)
    x['study_id']="sanchez2017chimpanzees"
    x = x.rename(columns={"sex_dyad": "dyad_sex",
        "dyad": "dyad_original",
        "subject left": "ape",
        "subject right": "ape_2"})
    x['role']='focal_participant_left'
    x['role_2']='focal_participant_right'
    x['ape'] = x['ape'].str.rstrip()
    x['ape_2'] = x['ape_2'].str.rstrip()
    data_frames[index]=x
new_df=data_frames[0]

fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)
    fulldf['ape_2'].replace(x, y, inplace=True)

fulldf['dyad']=fulldf.ape.str.cat(fulldf.ape_2, sep='_')
# fulldf.columns


In [102]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
fulldf= fulldf.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')


In [103]:
fulldf = fulldf.rename(columns={"species_y": "species",
    "sex_y": "sex",
    "food in non-social  ": "food_in_non_social",
    "position 5 rewards in social": "position_5_rewards_in_social_original",
    "position 1 reward in social": "position_1_reward_in_social_original",
    "choice subject right": "choice_participant_right_original",
    "choice subject left": "choice_participant_left_original",
    "pull latency subject right": "pull_latency_participant_right",
    "pull latency subject left": "pull_latency_participant_left",
    "get subject right": "get_participant_right",
    "get subject left": "get_participant_left",
    "pull (binomial)": "pull_binomial"})

fulldf.columns =fulldf.columns.str.replace(' ', '_')
# fulldf.columns

In [104]:
code_list=["position_1_reward_in_social_original", "position_5_rewards_in_social_original"]
for index, x in enumerate(code_list):    
    fulldf[x] = fulldf[x].astype(str)
    temp=[]
    for entry in fulldf[x]:
        if entry == 'in':
            entry = "roped_interior_side"
        elif entry =='out':
            entry = "free_exterior_side"
        temp.append(entry)
    fulldf = fulldf.assign(temp_col=temp)
    fulldf=fulldf.rename(columns={'temp_col': x+'_codes'})
fulldf =fulldf.rename(columns={"position_1_reward_in_social_original_codes":"position_1_reward_in_social",
                               "position_5_rewards_in_social_original_codes":"position_5_rewards_in_social"})

In [105]:
code_list=["choice_participant_right_original", "choice_participant_left_original"]
for index, x in enumerate(code_list):    
    fulldf[x] = fulldf[x].astype(str)
    temp=[]
    for entry in fulldf[x]:
        if entry == 'soc':
            entry = "social"
        elif entry =='non_soc':
            entry = "non_social"
        elif entry =='no':
            entry = "no_choice"
        temp.append(entry)
    fulldf = fulldf.assign(temp_col=temp)
    fulldf=fulldf.rename(columns={'temp_col': x+'_codes'})
fulldf =fulldf.rename(columns={"choice_participant_right_original_codes":"choice_participant_right",
                               "choice_participant_left_original_codes":"choice_participant_left"})

In [106]:
fulldf =fulldf.rename(columns={"pull_binomial":"pull_binomial_original"})
code_list=["pull_binomial_original"]
for index, x in enumerate(code_list):    
    fulldf[x] = fulldf[x].astype(str)
    temp=[]
    for entry in fulldf[x]:
        if entry == '1' or entry=='1.0':
            entry = "both_pull"
        elif entry =='0' or entry=='0.0':
            entry = "both_did_not_pull"
        temp.append(entry)
    fulldf = fulldf.assign(temp_col=temp)
    fulldf=fulldf.rename(columns={'temp_col': 'pull_binomial'})

In [107]:
fulldf =fulldf.rename(columns={"condition":"condition_original"})
code_list=["condition_original"]
for index, x in enumerate(code_list):    
    fulldf[x] = fulldf[x].astype(str)
    temp=[]
    for entry in fulldf[x]:
        if entry == 'sd':
            entry = "social_dilemma"
        elif entry =='com':
            entry = "competition"
        temp.append(entry)
    fulldf = fulldf.assign(temp_col=temp)
    fulldf=fulldf.rename(columns={'temp_col': 'condition'})

In [108]:
fulldf.rename(columns={"subject_left_pull_latency": "participant_left_pull_latency", 
                    "subject_right_pull_latency":"participant_right_pull_latency"}, inplace=True)

replace_list_1 = ['participant_left_pull_latency','participant_right_pull_latency', 
                    'pull_latency_participant_right', 'pull_latency_participant_left']
for x in replace_list_1:
    fulldf[x].replace('np', 'no_pull', inplace=True, regex=True)

replace_list_2 = ['latency_open_door_right', 'latency_open_door_left']
for x in replace_list_2:
    fulldf[x].replace('n', 'door_not_opened', inplace=True, regex=True)



In [109]:
fulldf.replace('nan', np.nan, inplace=True)
fulldf.rename(columns={"ape": "participant", "ape_2":"participant_2",
                       'participant_left_pull_latency':"focal_participant_left_pull_latency", 
                       'participant_right_pull_latency':'focal_participant_right_pull_latency',
                       'choice_participant_right':'choice_focal_participant_right',
                       'choice_participant_left':'choice_focal_participant_left',
                       'pull_latency_participant_right':'pull_latency_focal_participant_right',
                       'pull_latency_participant_left':'pull_latency_focal_participant_left', 
                       'get_participant_right':'get_focal_participant_right',
                         'get_participant_left':'get_focal_participant_left'}, inplace=True)

comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left')

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
fulldf= fulldf.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')
two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    fulldf[x] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
    fulldf[x] = pd.to_datetime(fulldf[x])
    fulldf[y] = pd.to_datetime(fulldf[y])
    fulldf[k] = (fulldf[x] - fulldf[y]).dt.days//365

In [110]:
fulldf=fulldf[['study_id', 'experiment_name','year','month','day', 
         'participant',  'age_in_years','sex', 'role',
         'participant_2', 'age_in_years_2','sex_2', 'role_2', 'species', 'dyad', 'dyad_sex',  
       'phase', 'session', 'trial',
       'condition', 'pull', 
       'focal_participant_left_pull_latency', 'focal_participant_right_pull_latency',
       'get_4', 'get_1',
       'minimum_latency_to_pull', 'pull_binomial',
       'food_in_non_social', 'position_5_rewards_in_social',
       'position_1_reward_in_social', 
       'choice_focal_participant_right','choice_focal_participant_left',
         'latency_open_door_right',
       'latency_open_door_left', 
       'pull_latency_focal_participant_right','pull_latency_focal_participant_left', 'get_focal_participant_right', 'get_focal_participant_left' ]]


In [111]:
exp1 = fulldf[fulldf['experiment_name'] == '1a']
exp2 = fulldf[fulldf['experiment_name'] == '2']

experiments = [[exp1, 'sanchez2017chimpanzees_exp1a'],
                [ exp2, 'sanchez2017chimpanzees_exp2']]

for x,y in experiments:
    x = x.dropna(axis=1, how='all')## drop empty rows/columns
    comp_out_path_stand = os.path.join(out_pathway, y+'_standardized.csv')
    x.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)
    ##glossaries
    names = x.columns.tolist()
    df = pd.DataFrame(names)
    df = df.rename(columns={0: "column_name"})
    df["description"] = ""
    studyID_glossary=df[["column_name", "description"]]

    comp_out_path_glossary = os.path.join(out_pathway, y+'_glossary.csv')
    studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)